In [ ]:
!tar -xvf 'laion400m-data\00000.tar'
# !tar -xvf '/content/00001.tar'
# !tar -xvf '/content/00002.tar'

: 

In [2]:
import torch
import torchvision.transforms as transforms
import os
from PIL import Image, ImageFile

In [3]:
# 손상된 이미지 로딩 허용
ImageFile.LOAD_TRUNCATED_IMAGES = True

input_folder = "/content/"  # 압축 풀린 이미지 폴더
output_folder = "resized_images"  # 리사이즈된 이미지 저장 폴더

os.makedirs(output_folder, exist_ok=True)

for img_name in os.listdir(input_folder):
    if img_name.lower().endswith(".jpg"):  # JPG 파일만 처리
        img_path = os.path.join(input_folder, img_name)

        try:
            img = Image.open(img_path)
            img_resized = img.resize((224, 224))
            img_resized.save(os.path.join(output_folder, img_name))
        except OSError as e:
            print(f"🚨 {img_name} 변환 실패: {e}")

print("224x224로 리사이즈 완료!")

224x224로 리사이즈 완료!


In [4]:
# 이미지 변환 파이프라인 (Tensor 변환 + 정규화)
transform = transforms.Compose([
    transforms.ToTensor(),  # PIL Image → Tensor 변환 (값 범위: [0,1])
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet 정규화
])

def image_to_tensor(image_path):
    """이미지를 PyTorch Tensor로 변환"""
    img = Image.open(image_path).convert("RGB")  # RGB 변환
    img_tensor = transform(img)  # Tensor 변환 + 정규화
    return img_tensor


In [6]:
input_folder = "resized_images"  # 224x224로 리사이즈된 이미지 폴더
image_tensors = []

for img_name in os.listdir(input_folder):
    if img_name.lower().endswith(".jpg"):
        img_path = os.path.join(input_folder, img_name)
        img_tensor = image_to_tensor(img_path)
        image_tensors.append(img_tensor)

# List of tensors → Single tensor (Batch 형태)
image_tensors = torch.stack(image_tensors)

print("변환 완료! Shape:", image_tensors.shape)  # (Batch, C, H, W)


변환 완료! Shape: torch.Size([79, 3, 224, 224])


In [7]:
import torchvision.models as models

# ResNet-50 모델 (사전 훈련된 버전)
resnet = models.resnet50(pretrained=True)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])  # 마지막 FC Layer 제거 (Feature Extractor)

# 모델을 평가 모드로 설정
resnet.eval()

# 표현 벡터 추출
with torch.no_grad():
    feature_vectors = resnet(image_tensors)  # (N, 2048, 1, 1)
    feature_vectors = feature_vectors.squeeze(-1).squeeze(-1)  # (N, 2048)

print("표현 벡터 추출 완료! Shape:", feature_vectors.shape)


표현 벡터 추출 완료! Shape: torch.Size([79, 2048])
